In [1]:
!pip install sentence-transformers faiss-cpu chromadb markdown

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.4/483.4 KB 2.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 3.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 3.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 KB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 2.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 KB 3.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 3.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 3.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 2.4 MB/s eta 0:00:0000:0100:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import markdown
from sentence_transformers import SentenceTransformer
import numpy as np

# Load markdown file
with open('scraped_content.md', 'r', encoding='utf-8') as f:
    md_content = f.read()

# Convert markdown to plain text (optional: keeps structure but removes HTML tags)
plain_text = markdown.markdown(md_content)

# Preprocess: Split into chunks (e.g., paragraphs)
chunks = [chunk.strip() for chunk in plain_text.split('\n\n') if chunk.strip()]

print(f"Number of chunks: {len(chunks)}")
print("Sample chunk:", chunks[0] if chunks else "No content")

In [ ]:
# Load pre-trained model (you can swap to others like 'all-mpnet-base-v2' for better quality but slower)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings (list of numpy arrays)
embeddings = model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")  # e.g., (num_chunks, 384)

In [ ]:
import faiss

# Create FAISS index (dimension from embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)  # L2 distance (Euclidean)

# Add embeddings to index
index.add(embeddings)

# Save index to disk (optional)
faiss.write_index(index, 'faiss_index.index')

# To query later (example: find top 3 similar to a query)
query_text = "Your search query here"
query_emb = model.encode([query_text])
D, I = index.search(query_emb, k=3)  # D: distances, I: indices

# Retrieve results
for idx in I[0]:
    print(chunks[idx])